# MLB Statcast Analytics Pipeline

This notebook builds the Databricks data model used by the Tableau MLB Pitcher Profile dashboard.

**Pipeline:** Statcast → Bronze → Silver → Gold → Tableau reporting views

**Analysis window:** April 1–7, 2025

The notebook is intentionally organized as a reproducible pipeline rather than an exploratory scratchpad. Each section creates or validates one layer of the model.


## 1. Environment and schema setup

Install the Statcast extraction package, import the libraries used throughout the notebook, and ensure the `mlb` schema exists.


In [0]:
%pip install pybaseball
dbutils.library.restartPython()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.3/818.3 kB 40.8 MB/s eta 0:00:00
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Not uninstalling cffi at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-55943bf9-b604-4b60-a42b-97115c195af7
    Can't uninstall 'cffi'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pybaseball import statcast
import pandas as pd
import requests
import time

from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS mlb")

spark.sql(
    "SELECT current_catalog() AS current_catalog, current_schema() AS current_schema"
).show()


+---------------+--------------+
|current_catalog|current_schema|
+---------------+--------------+
|      workspace|       default|
+---------------+--------------+



## 2. Pipeline parameters

Keeping the analysis window in one place makes the notebook easier to rerun or extend later.


In [0]:
START_DATE = "2025-04-01"
END_DATE = "2025-04-07"

print(f"Statcast window: {START_DATE} to {END_DATE}")


Statcast window: 2025-04-01 to 2025-04-07


## 3. Bronze layer — raw Statcast ingestion

Pull the requested Statcast date range with `pybaseball` and store the full raw response in `mlb.statcast_bronze`.

Because this project uses a fixed historical window, `createOrReplace()` keeps the notebook idempotent: rerunning the notebook refreshes the Bronze table without duplicating pitches.


In [0]:
pitches_pd = statcast(
    start_dt=START_DATE,
    end_dt=END_DATE
)

if pitches_pd.empty:
    raise ValueError("Statcast returned no rows for the requested date range.")

print(f"Rows extracted: {len(pitches_pd):,}")
display(pitches_pd.head())


This is a large query, it may take a moment to complete


  0%|          | 0/7 [00:00<?, ?it/s]/local_disk0/.ephemeral_nfs/envs/pythonEnv-55943bf9-b604-4b60-a42b-97115c195af7/lib/python3.12/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-55943bf9-b604-4b60-a42b-97115c195af7/lib/python3.12/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-55943bf9-b604-4b60-a42b-97115c195af7/lib/python3.12/site-packages/pybaseball/datahelpers/p

Rows extracted: 25,477


pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,miss_distance,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
FF,2025-04-07T00:00:00.000Z,95.4,-2.65,5.79,"Hoffman, Jeff",666152,656546,field_out,hit_into_play,null,null,null,null,3,David Hamilton flies out to left fielder Alan Roden.,R,L,R,BOS,TOR,X,7,fly_ball,0,0,2025,-0.95,1.61,0.6047783189835629,2.740227881272169,null,null,null,2,9,Bot,97.09,123.31,null,null,null,null,10.736929446018063,-138.2174312801077,-5.877537805407462,-14.844318590388358,36.46809082658631,-10.4831164656062,3.18,1.45,181,93.0,62,94.9,2548,6.6,778426,672386,665489,665926,676391,666182,702176,664702,543807,53.86,0.018,0.02,0.0,1,0,0,3,79,1,4-Seam Fastball,2,6,2,6,6,2,2,6,Infield shade,Standard,222,-0.001,-0.26,73.9,7.0,null,0.028,0.26,93.0,-4,-4,0.001,0.001,32,27,32,28,1,3,2,1,2,2,0.96,0.95,-0.95,34.8,8.967642715300745,-8.637136410958288,34.274898058624565,31.03833992954139,35.393402710400466
FS,2025-04-07T00:00:00.000Z,90.0,-2.6,5.71,"Hoffman, Jeff",665966,656546,field_out,hit_into_play,null,null,null,null,14,"Carlos Narváez grounds out softly, third baseman Ernie Clement to first baseman Vladimir Guerrero Jr.",R,R,R,BOS,TOR,X,5,ground_ball,1,1,2025,-1.53,0.38,0.3637609340195831,1.489470645488668,null,null,null,1,9,Bot,108.05,184.73,null,null,null,null,10.659807922216348,-130.54931021084022,-5.273263825958916,-19.82644100259839,29.131886652859546,-26.788061116168024,3.54,1.62,2,73.4,-43,90.2,1651,6.7,778426,672386,665489,665926,676391,666182,702176,664702,543807,53.77,0.13,0.12,0.0,1,0,0,2,78,3,Split-Finger,2,6,2,6,6,2,2,6,Standard,Standard,235,-0.006,-0.244,70.7,8.6,null,0.13,0.244,88.0,-4,-4,0.007,0.007,32,26,32,27,1,3,2,1,2,1,2.46,1.53,1.53,35.6,9.84789225417792,-1.050836434035521,28.63663681246836,45.98229995833792,31.944116596804275
FF,2025-04-07T00:00:00.000Z,95.8,-2.6,5.78,"Hoffman, Jeff",665966,656546,null,ball,null,null,null,null,14,null,R,R,R,BOS,TOR,B,null,null,0,1,2025,-1.01,1.48,1.065296048244752,1.773555464601018,null,null,null,1,9,Bot,null,null,null,null,null,null,11.998990916168088,-138.65120411195193,-8.240873507725858,-15.950028405262191,35.29189187196722,-11.231262452127783,3.5831888599577,1.61952609336281,null,null,null,95.4,2559,6.7,778426,672386,665489,665926,676391,666182,702176,664702,543807,53.81,null,null,null,null,null,null,null,78,2,4-Seam Fastball,2,6,2,6,6,2,2,6,Standard,Standard,217,0.001,0.029,null,null,null,null,-0.029,null,-4,-4,0.006,0.006,32,26,32,27,1,3,2,1,2,1,1.06,1.01,1.01,42.8,null,null,null,null,null
FF,2025-04-07T00:

In [0]:
pitches_spark = spark.createDataFrame(pitches_pd)

pitches_spark.writeTo("mlb.statcast_bronze").createOrReplace()

print(f"Bronze rows: {spark.table('mlb.statcast_bronze').count():,}")


Bronze rows: 25,477


In [0]:
# Validate that the expected dates were ingested.
(
    spark.table("mlb.statcast_bronze")
    .groupBy("game_date")
    .count()
    .orderBy("game_date")
    .show()
)


+-------------------+-----+
|          game_date|count|
+-------------------+-----+
|2025-04-01 00:00:00| 3704|
|2025-04-02 00:00:00| 4295|
|2025-04-03 00:00:00| 1451|
|2025-04-04 00:00:00| 4385|
|2025-04-05 00:00:00| 4012|
|2025-04-06 00:00:00| 4667|
|2025-04-07 00:00:00| 2963|
+-------------------+-----+



## 4. Silver layer — dashboard-ready pitch data

Select the fields required downstream, remove rows without a classified pitch type, and create a stable pitch identifier from Statcast's natural pitch key:

`game_pk + at_bat_number + pitch_number`

The natural key is preferable to a hash of descriptive pitch attributes because it directly identifies one pitch within one game.


In [0]:
silver = (
    spark.table("mlb.statcast_bronze")
    .select(
        "game_date",
        "game_pk",
        "at_bat_number",
        "pitch_number",
        "player_name",
        "pitcher",
        "batter",
        "pitch_type",
        "release_speed",
        "plate_x",
        "plate_z",
        "balls",
        "strikes",
        "description",
        "events",
        "home_team",
        "away_team"
    )
    .filter(F.col("pitch_type").isNotNull())
    .withColumn(
        "pitch_id",
        F.concat_ws(
            "-",
            F.col("game_pk").cast("string"),
            F.col("at_bat_number").cast("string"),
            F.col("pitch_number").cast("string")
        )
    )
)

silver.writeTo("mlb.statcast_silver").createOrReplace()

print(f"Silver rows: {spark.table('mlb.statcast_silver').count():,}")


Silver rows: 25,358


In [0]:
# Critical Silver-layer data-quality checks.
silver_check = spark.table("mlb.statcast_silver")

quality_summary = silver_check.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("pitch_id").alias("unique_pitch_ids"),
    F.sum(F.col("plate_x").isNull().cast("int")).alias("null_plate_x"),
    F.sum(F.col("plate_z").isNull().cast("int")).alias("null_plate_z"),
    F.sum(F.col("release_speed").isNull().cast("int")).alias("null_release_speed")
)

display(quality_summary)


total_rows,unique_pitch_ids,null_plate_x,null_plate_z,null_release_speed
25358,25358,0,0,0


## 5. Player dimension — MLB profile enrichment

Build one player dimension containing the pitchers and batters present in the Silver data. MLB Stats API enriches each player ID with profile fields used by Tableau, including handedness, height, weight, debut date, birthplace, and headshot URL.

The dashboard currently uses the pitcher profile fields, while retaining batters in the dimension keeps the model extensible.


In [0]:
player_ids = (
    spark.table("mlb.statcast_silver")
    .select(F.col("pitcher").alias("player_id"))
    .union(
        spark.table("mlb.statcast_silver")
        .select(F.col("batter").alias("player_id"))
    )
    .filter(F.col("player_id").isNotNull())
    .distinct()
)

player_id_list = [row["player_id"] for row in player_ids.collect()]

print(f"Unique players to enrich: {len(player_id_list):,}")


Unique players to enrich: 790


In [0]:
session = requests.Session()
player_records = []

for player_id in player_id_list:
    url = f"https://statsapi.mlb.com/api/v1/people/{player_id}"

    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()
        people = response.json().get("people", [])

        if not people:
            continue

        p = people[0]

        player_records.append({
            "player_id": p.get("id"),
            "full_name": p.get("fullName"),
            "first_name": p.get("firstName"),
            "last_name": p.get("lastName"),
            "position": p.get("primaryPosition", {}).get("name"),
            "position_abbr": p.get("primaryPosition", {}).get("abbreviation"),
            "bats": p.get("batSide", {}).get("code"),
            "throws": p.get("pitchHand", {}).get("code"),
            "height": p.get("height"),
            "weight": p.get("weight"),
            "birth_date": p.get("birthDate"),
            "birth_country": p.get("birthCountry"),
            "mlb_debut_date": p.get("mlbDebutDate"),
            "active": p.get("active"),
            "headshot_url": (
                "https://img.mlbstatic.com/mlb-photos/image/upload/"
                "d_people:generic:headshot:67:current.png/"
                "w_213,q_auto:best/"
                f"v1/people/{player_id}/headshot/67/current.png"
            )
        })

    except requests.RequestException as exc:
        print(f"Player {player_id}: API request failed ({exc})")

    # Small pause to avoid sending requests in a tight loop.
    time.sleep(0.05)

players_enriched_pd = pd.DataFrame(player_records)

print(f"Player profiles returned: {len(players_enriched_pd):,}")
display(players_enriched_pd.head(10))


Player profiles returned: 790


player_id,full_name,first_name,last_name,position,position_abbr,bats,throws,height,weight,birth_date,birth_country,mlb_debut_date,active,headshot_url
595978,Austin Hedges,Austin,Hedges,Catcher,C,R,R,"6' 0""",220,1992-08-18,USA,2015-05-04,true,"https://img.mlbstatic.com/mlb-photos/image/upload/d_people:generic:headshot:67:current.png/w_213,q_auto:best/v1/people/595978/headshot/67/current.png"
695243,Mason Miller,Mason,Miller,Pitcher,P,R,R,"6' 5""",200,1998-08-24,USA,2023-04-19,true,"https://img.mlbstatic.com/mlb-photos/image/upload/d_people:generic:headshot:67:current.png/w_213,q_auto:best/v1/people/695243/headshot/67/current.png"
702616,Jackson Holliday,Jackson,Holliday,Second Base,2B,L,R,"5' 11""",200,2003-12-04,USA,2024-04-10,true,"https://img.mlbstatic.com/mlb-photos/image/upload/d_people:generic:headshot:67:current.png/w_213,q_auto:best/v1/people/702616/headshot/67/current.png"
624424,Michael Conforto,Michael,Conforto,Designated Hitter,DH,L,R,"5' 11""",215,1993-03-01,USA,2015-07-24,true,"https://img.mlbstatic.com/mlb-photos/image/upload/d_people:generic:headshot:67:current.png/w_213,q_auto:best/v1/people/624424/headshot/67/current.png"
663767,Chase Shugart,Chase,Shugart,Pitcher,P,L,R,"5' 11""",200,1996-10-24,USA,2024-08-15,true,"https://img.mlbstatic.com/mlb-photos/image/upload/d_people:generic:headshot:67:current.png/w_213,q_auto:best/v1/people/663767/headshot/67/current.png"
596103,Austin Slater,Austin,Slater,Outfielder,RF,R,R,"6' 1""",205,1992-12-13,USA,2017-06-02,true,"https://img.mlbstatic.com/mlb-photos/image/upload/d_people:generic:headshot:67:current.png/w_213,q_auto:best/v1/people/596103/headshot/67/current.png"
518595,Travis d'Arnaud,Travis,d'Arnaud,Catcher,C,R,R,"6' 0""",210,1989-02-10,USA,2013-08-17,true,"https://img.mlbstatic.com/mlb-photos/image/upload/d_people:generic:headshot:67:current.png/w_213,q_auto:best/v1/people/518595/headshot/67/current.png"
683409,Angel Chivilli,Angel,Chivilli,Pitcher,P,R,R,"6' 2""",185,2002-07-28,Dominican Republic,2024-06-03,true,"https://img.mlbstatic.com/mlb-photos/image/upload/d_people:generic:headshot:67:current.png/w_213,q_auto:best/v1/people/683409/headshot/67/current.png"
656546,Jeff Hoffman,Jeffrey,Hoffman,Pitcher,P,R,R,"6' 5""",235,1993-01-08,USA,2016-08-20,true,"https://img.mlbstatic.com/mlb-photos/image/upload/d_people:generic:headshot:67:current.png/w_213,q_auto:best/v1/people/656546/headshot/67/current.png"
656302,Dylan Cease,Dylan,Cease,Pitcher,P,R,R,"6' 2""",200,1995-12-28,USA,2019-07-03,true,"https://img.mlbstatic.com/mlb-photos/image/upload/d_people:generic:headshot:67:current.png/w_213,q_auto:best/v1/people/656302/headshot/67/current.png"


In [0]:
if players_enriched_pd.empty:
    raise ValueError("No player profiles were returned by MLB Stats API.")

dim_player = (
    spark.createDataFrame(players_enriched_pd)
    .withColumn("birth_date", F.to_date("birth_date"))
    .withColumn("mlb_debut_date", F.to_date("mlb_debut_date"))
    .dropDuplicates(["player_id"])
)

dim_player.writeTo("mlb.dim_player").createOrReplace()

print(f"Player dimension rows: {spark.table('mlb.dim_player').count():,}")


Player dimension rows: 790


## 6. Gold layer — pitcher arsenal

Aggregate Silver pitch-level data to one row per pitcher and pitch type. This table powers the Pitcher Arsenal and Usage vs. Velocity visuals in Tableau.

`usage_pct` is calculated within each pitcher, so a pitcher's pitch-type shares should sum to approximately 100%.


In [0]:
silver = spark.table("mlb.statcast_silver")

pitcher_arsenal = (
    silver
    .groupBy(
        "pitcher",
        "player_name",
        "pitch_type"
    )
    .agg(
        F.count("*").alias("pitch_count"),
        F.round(F.avg("release_speed"), 1).alias("avg_velocity")
    )
)

pitcher_window = Window.partitionBy("pitcher")

pitcher_arsenal = (
    pitcher_arsenal
    .withColumn(
        "total_pitches",
        F.sum("pitch_count").over(pitcher_window)
    )
    .withColumn(
        "usage_pct",
        F.round(
            F.col("pitch_count") / F.col("total_pitches") * 100,
            1
        )
    )
)

pitcher_arsenal.writeTo("mlb.pitcher_arsenal_gold").createOrReplace()

display(
    pitcher_arsenal
    .orderBy(F.desc("total_pitches"), F.desc("usage_pct"))
    .limit(20)
)


pitcher,player_name,pitch_type,pitch_count,avg_velocity,total_pitches,usage_pct
622663,"Severino, Luis",SI,63,95.0,203,31.0
622663,"Severino, Luis",FF,59,95.0,203,29.1
622663,"Severino, Luis",ST,47,84.1,203,23.2
622663,"Severino, Luis",FC,16,92.0,203,7.9
622663,"Severino, Luis",CH,9,85.6,203,4.4
622663,"Severino, Luis",SL,9,86.5,203,4.4
607074,"Rodón, Carlos",FF,91,93.1,199,45.7
607074,"Rodón, Carlos",SL,46,84.3,199,23.1
607074,"Rodón, Carlos",CH,22,85.4,199,11.1
607074,"Rodón, Carlos",SI,22,90.7,199,11.1


In [0]:
# The Gold pitch counts should reconcile to the Silver pitch-level table.
gold_pitch_total = (
    spark.table("mlb.pitcher_arsenal_gold")
    .agg(F.sum("pitch_count").alias("total"))
    .first()["total"]
)

silver_pitch_total = spark.table("mlb.statcast_silver").count()

print(f"Gold pitch count:   {gold_pitch_total:,}")
print(f"Silver pitch count: {silver_pitch_total:,}")

if gold_pitch_total != silver_pitch_total:
    raise ValueError("Gold and Silver pitch totals do not match.")


Gold pitch count:   25,358
Silver pitch count: 25,358


## 7. Gold layer — pitcher summary

Create one row per pitcher for the dashboard KPI cards. The primary pitch is defined as the pitch type with the highest observed pitch count in the analysis window.


In [0]:
%sql
CREATE OR REPLACE TABLE mlb.pitcher_summary_gold AS

WITH pitch_counts AS (
    SELECT
        pitcher,
        pitch_type,
        COUNT(*) AS pitch_count
    FROM mlb.statcast_silver
    GROUP BY pitcher, pitch_type
),

ranked_pitches AS (
    SELECT
        pitcher,
        pitch_type,
        pitch_count,
        ROW_NUMBER() OVER (
            PARTITION BY pitcher
            ORDER BY pitch_count DESC, pitch_type
        ) AS rn
    FROM pitch_counts
),

pitcher_summary AS (
    SELECT
        pitcher AS player_id,
        COUNT(*) AS total_pitches,
        ROUND(AVG(release_speed), 1) AS avg_velocity,
        COUNT(DISTINCT pitch_type) AS pitch_types
    FROM mlb.statcast_silver
    GROUP BY pitcher
)

SELECT
    s.player_id,
    s.total_pitches,
    r.pitch_type AS primary_pitch,
    s.avg_velocity,
    s.pitch_types
FROM pitcher_summary s
LEFT JOIN ranked_pitches r
    ON s.player_id = r.pitcher
   AND r.rn = 1;


num_affected_rows,num_inserted_rows


## 8. Tableau reporting views

Expose only the fields Tableau needs. Keeping the dashboard connected to reporting views separates presentation logic from the underlying Gold tables and makes the model easier to maintain.


In [0]:
%sql
CREATE OR REPLACE VIEW mlb.v_pitcher_arsenal_report AS
SELECT
    g.pitcher AS player_id,
    p.full_name AS player_name,
    p.position,
    p.position_abbr,
    p.bats,
    p.throws,
    p.height,
    p.weight,
    p.birth_date,
    p.birth_country,
    p.mlb_debut_date,
    p.headshot_url,
    g.pitch_type,
    g.pitch_count,
    g.avg_velocity,
    g.total_pitches,
    g.usage_pct
FROM mlb.pitcher_arsenal_gold g
LEFT JOIN mlb.dim_player p
    ON g.pitcher = p.player_id;


In [0]:
%sql
CREATE OR REPLACE VIEW mlb.v_pitcher_summary_report AS
SELECT
    s.player_id,
    p.full_name AS player_name,
    s.total_pitches,
    s.primary_pitch,
    s.avg_velocity,
    s.pitch_types
FROM mlb.pitcher_summary_gold s
LEFT JOIN mlb.dim_player p
    ON s.player_id = p.player_id;


In [0]:
%sql
CREATE OR REPLACE VIEW mlb.v_pitch_location_report AS
SELECT
    pitch_id,
    game_date,
    game_pk,
    player_name,
    pitcher,
    batter,
    pitch_type,
    release_speed,
    plate_x,
    plate_z,
    balls,
    strikes,
    description,
    events,
    home_team,
    away_team
FROM mlb.statcast_silver
WHERE plate_x IS NOT NULL
  AND plate_z IS NOT NULL;


## 9. Final validation

Run compact checks on the objects exposed to Tableau. These checks verify pitch-level uniqueness, profile joins, pitch-location completeness, and profile-field coverage for pitchers available in the dashboard.


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT pitch_id) AS unique_pitch_ids
FROM mlb.v_pitch_location_report;


total_rows,unique_pitch_ids
25358,25358


In [0]:
%sql
SELECT
    COUNT(*) AS total_pitches,
    SUM(CASE WHEN plate_x IS NULL THEN 1 ELSE 0 END) AS null_plate_x,
    SUM(CASE WHEN plate_z IS NULL THEN 1 ELSE 0 END) AS null_plate_z,
    SUM(
        CASE
            WHEN plate_x IS NULL OR plate_z IS NULL THEN 1
            ELSE 0
        END
    ) AS missing_either
FROM mlb.statcast_silver;


total_pitches,null_plate_x,null_plate_z,missing_either
25358,0,0,0


In [0]:
%sql
SELECT
    COUNT(DISTINCT player_id) AS total_pitchers,
    COUNT(DISTINCT CASE WHEN headshot_url IS NULL OR TRIM(headshot_url) = '' THEN player_id END) AS missing_photo,
    COUNT(DISTINCT CASE WHEN player_name IS NULL THEN player_id END) AS missing_name,
    COUNT(DISTINCT CASE WHEN bats IS NULL THEN player_id END) AS missing_bats,
    COUNT(DISTINCT CASE WHEN throws IS NULL THEN player_id END) AS missing_throws,
    COUNT(DISTINCT CASE WHEN height IS NULL THEN player_id END) AS missing_height,
    COUNT(DISTINCT CASE WHEN weight IS NULL THEN player_id END) AS missing_weight,
    COUNT(DISTINCT CASE WHEN birth_date IS NULL THEN player_id END) AS missing_birth_date,
    COUNT(DISTINCT CASE WHEN birth_country IS NULL THEN player_id END) AS missing_birth_country,
    COUNT(DISTINCT CASE WHEN mlb_debut_date IS NULL THEN player_id END) AS missing_debut,
    COUNT(DISTINCT CASE WHEN position IS NULL THEN player_id END) AS missing_position
FROM mlb.v_pitcher_arsenal_report;


total_pitchers,missing_photo,missing_name,missing_bats,missing_throws,missing_height,missing_weight,missing_birth_date,missing_birth_country,missing_debut,missing_position
399,0,0,0,0,0,0,0,0,0,0


In [0]:
%sql
SELECT DISTINCT
    player_id,
    player_name,
    headshot_url,
    bats,
    throws,
    height,
    weight,
    birth_date,
    birth_country,
    mlb_debut_date,
    position
FROM mlb.v_pitcher_arsenal_report
WHERE headshot_url IS NULL
   OR TRIM(headshot_url) = ''
   OR player_name IS NULL
   OR bats IS NULL
   OR throws IS NULL
   OR height IS NULL
   OR weight IS NULL
   OR birth_date IS NULL
   OR birth_country IS NULL
   OR mlb_debut_date IS NULL
   OR position IS NULL
ORDER BY player_name;


player_id,player_name,headshot_url,bats,throws,height,weight,birth_date,birth_country,mlb_debut_date,position


## 10. Tableau model reference

The Tableau workbook uses these reporting views:

- **`mlb.v_pitcher_summary_report`** — KPI cards such as total pitches, primary pitch, average velocity, and pitch-type count.
- **`mlb.v_pitcher_arsenal_report`** — global pitcher selector, player profile, Pitcher Arsenal, and Usage vs. Velocity.
- **`mlb.v_pitch_location_report`** — Pitch Location and Pitch Results.

For the published dashboard, a Tableau **Extract** is appropriate because this is a fixed historical analysis window and it provides much faster interaction than repeated live Databricks queries.
